# Prerequisites
Before we begin, we need to install several key libraries required for natural language processing, data manipulation, and model training.

In [ ]:
!pip install transformers datasets pandas scikit-learn accelerate

## Dataset Introduction
For this project, we are using a specialized customer complaints dataset. You can access the raw data here:
[Download Dataset](https://drive.google.com/file/d/12zqr1ltiy8jLPFqp5DAWHLA0_mIlSeed/view?usp=sharing)

This dataset is essential for training our hybrid system as it contains labeled customer feedback with specific fields for:
* **text**: The raw customer complaint.
* **sentiment**: The emotional tone (Negative, Neutral, Positive).
* **urgency**: The priority level (Low, Medium, High).

# Hybrid Customer Complaint Analysis System
This notebook demonstrates a multi-model approach to analyzing customer complaints using DistilBERT for NLP and rule-based logic for safety and sarcasm detection.

In [ ]:
!pip install transformers datasets pandas scikit-learn accelerate


## 1. Environment Setup
We begin by installing the necessary Hugging Face and data science libraries.
###Libraries used
To build this hybrid system, we rely on a specific stack of open-source tools:

*   **`transformers` (Hugging Face)**: Provides the pre-trained DistilBERT architecture and high-level APIs for fine-tuning NLP models.
*   **`datasets`**: Optimized for handling large-scale text data and efficient tokenization batches.
*   **`pandas` & `numpy`**: The standard toolkit for data manipulation, CSV loading, and numerical vector operations.
*   **`scikit-learn`**: Used for data preprocessing (like `LabelEncoder`) and splitting our dataset into training/validation sets.
*   **`accelerate`**: A utility library that handles the heavy lifting of running PyTorch models on GPU (T4) hardware without complex boilerplate code.
*   **`torch` (PyTorch)**: The underlying deep learning framework used to calculate losses and run the neural network layers.

In [ ]:
import pandas as pd
import numpy as np
import ast
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Quick setup for the model and tokenizer we'll be using throughout
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Defining our business categories here
ALL_CATEGORIES = [
    'billing',
    'shipping',
    'product_quality',
    'customer_service',
    'technical_support',
    'account_access',
    'refund_request',
    'safety_issue',
    'order_status',
]

# Helper to handle the text tokenization consistently
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length",
                     truncation=True, max_length=128)

# Pulling in our data and doing some quick cleanup
df = pd.read_csv("complaints_dataset.csv")

# Let's turn those category strings back into actual lists for the model
df['categories_vector'] = df['categories_vector'].apply(
    lambda x: [float(v) for v in ast.literal_eval(x)]
)

print(f"Loaded {len(df)} rows")
display(df[['text', 'sentiment', 'urgency', 'categories_str']].head(5))

# Setting up the Sentiment model training
print("\n" + "="*50)
print("TRAINING MODEL 1: SENTIMENT")
print("="*50)

le_sentiment = LabelEncoder()
df['sentiment_label'] = le_sentiment.fit_transform(df['sentiment'])

# Splitting our data so we can validate how the model is learning
train_df, test_df = train_test_split(
    df[['text', 'sentiment_label']], test_size=0.2, random_state=42
)
train_ds = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
test_ds  = Dataset.from_pandas(test_df ).map(tokenize_function, batched=True)
train_ds = train_ds.rename_column("sentiment_label", "labels")
test_ds  = test_ds.rename_column("sentiment_label", "labels")

model_s = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le_sentiment.classes_)
)

def compute_accuracy(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).astype(np.float32).mean().item()}

args_s = TrainingArguments(
    output_dir="./results_sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)
Trainer(model=model_s, args=args_s,
        train_dataset=train_ds, eval_dataset=test_ds,
        compute_metrics=compute_accuracy).train()
model_s.save_pretrained("./saved_model_sentiment")
tokenizer.save_pretrained("./saved_model_sentiment")

# Now let's handle the Urgency model
print("\n" + "="*50)
print("TRAINING MODEL 2: URGENCY")
print("="*50)

le_urgency = LabelEncoder()
df['urgency_label'] = le_urgency.fit_transform(df['urgency'])

train_df, test_df = train_test_split(
    df[['text', 'urgency_label']], test_size=0.2, random_state=42
)
train_ds = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
test_ds  = Dataset.from_pandas(test_df ).map(tokenize_function, batched=True)
train_ds = train_ds.rename_column("urgency_label", "labels")
test_ds  = test_ds.rename_column("urgency_label", "labels")

model_u = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le_urgency.classes_)
)
args_u = TrainingArguments(
    output_dir="./results_urgency",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)
Trainer(model=model_u, args=args_u,
        train_dataset=train_ds, eval_dataset=test_ds,
        compute_metrics=compute_accuracy).train()
model_u.save_pretrained("./saved_model_urgency")
tokenizer.save_pretrained("./saved_model_urgency")

# Finally, the Multi-Label Category model
print("\n" + "="*50)
print("TRAINING MODEL 3: CATEGORY")
print("="*50)

train_df, test_df = train_test_split(
    df[['text', 'categories_vector']], test_size=0.2, random_state=42
)

train_ds = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
test_ds  = Dataset.from_pandas(test_df ).map(tokenize_function, batched=True)
train_ds = train_ds.rename_column("categories_vector", "labels")
test_ds  = test_ds.rename_column("categories_vector", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format ("torch", columns=["input_ids", "attention_mask", "labels"])

# A custom class to handle multiple labels at once using BCE loss
class MultiLabelDistilBert(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels, ignore_mismatched_sizes=True
        )

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels.float())
        return (loss, logits) if loss is not None else logits

model_c = MultiLabelDistilBert(num_labels=len(ALL_CATEGORIES))

def compute_multilabel_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.5).astype(int)
    exact_match = (preds == labels.astype(int)).all(axis=1).mean()
    return {"exact_match_accuracy": float(exact_match)}

args_c = TrainingArguments(
    output_dir="./results_category",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="exact_match_accuracy",
    report_to="none",
)

Trainer(model=model_c, args=args_c, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_multilabel_metrics).train()
model_c.bert.save_pretrained("./saved_model_category")
tokenizer.save_pretrained("./saved_model_category")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded 4000 rows
                                                text sentiment urgency  \
0  I opened the box and the screen on the charger...  negative    high   
1  The keyboard works fine mostly, except it lite...  negative    high   
2  Two-factor authentication is not sending me th...  negative  medium   
3  The tablet emitted an electric shock when I to...  negative    high   
4  I was promised a full refund but was charged a...  negative    high   

                            categories_str  
0                          product_quality  
1             product_quality,safety_issue  
2                           account_access  
3                             safety_issue  
4  billing,customer_service,refund_request  

TRAINING MODEL 1: SENTIMENT
Sentiment classes: ['negative', 'neutral', 'positive']


Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 2. Model Training & System Architecture

This section handles the core development of our intelligent analysis engine. The project is built across **7 distinct development phases** to ensure a robust, production-ready system:

### Implementation Roadmap
*   **Phase 1: Safety Override Rules** – Heuristic checks for physical danger keywords.
*   **Phase 2: Sarcasm Detection** – Contextual analysis to detect linguistic contradictions.
*   **Phase 3: Multi-Label Category Output** – Utilizing Sigmoid activation and BCE loss for overlapping business domains.
*   **Phase 4: Hybrid Pipeline** – Integrating heuristic rules with transformer-based deep learning.
*   **Phase 5: Expanded Categorization** – Supporting 9 distinct business labels.
*   **Phase 6: Confidence Scoring** – Providing transparency via probability distributions for all predictions.
*   **Phase 7: Artifact Packaging** – Versioning and zipping models for external deployment.

### Model Training Details
We fine-tune three separate DistilBERT architectures:
1.  **Sentiment Classifier**: 3-class (Negative, Neutral, Positive).
2.  **Urgency Classifier**: 3-class (Low, Medium, High).
3.  **Category Classifier**: 9-label Multi-label classification.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Keeping track of our labels for consistent lookups
ALL_CATEGORIES = ['billing', 'shipping', 'product_quality', 'customer_service', 'technical_support', 'account_access', 'refund_request', 'safety_issue', 'order_status']
SENTIMENT_LABELS = ['negative', 'neutral', 'positive']
URGENCY_LABELS   = ['high', 'low', 'medium']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Safety Check: If we see danger signs, we jump straight to 'High' urgency
SAFETY_KEYWORDS = ["smoke", "fire", "burning", "burn", "sparks", "exploded", "overheated", "swelling", "unsafe", "electric shock", "leaking", "melting", "scorched", "chemical smell"]

def apply_safety_override(text: str, urgency: str) -> str:
    text_lower = text.lower()
    if any(k in text_lower for k in SAFETY_KEYWORDS):
        return "high"
    return urgency

# Sarcasm Check: Catching cases where polite words hide real frustration
def detect_sarcasm(text: str) -> bool:
    text_lower = text.lower()
    pos = ["great", "amazing", "thanks", "wonderful", "excellent"]
    neg = ["late", "broken", "charged twice", "refund", "no response", "crashed"]
    return any(p in text_lower for p in pos) and any(n in text_lower for n in neg)

# Let's get our models loaded onto the GPU
models = {}
tokenizers = {}
model_paths = {'sentiment': './saved_model_sentiment', 'urgency': './saved_model_urgency', 'category': './saved_model_category'}

for name, path in model_paths.items():
    tokenizers[name] = AutoTokenizer.from_pretrained(path)
    models[name] = AutoModelForSequenceClassification.from_pretrained(path).to(device).eval()

# The main engine: Runs the text through ML, then applies our human-logic overrides
def analyze_complaint(text: str) -> dict:
    def run(m_name, t):
        inputs = tokenizers[m_name](t, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        with torch.no_grad(): return models[m_name](**inputs).logits

    # Getting the base predictions from the AI
    s_probs = torch.softmax(run('sentiment', text), dim=-1).cpu().numpy()[0]
    u_probs = torch.softmax(run('urgency', text), dim=-1).cpu().numpy()[0]
    c_probs = torch.sigmoid(run('category', text)).cpu().numpy()[0]

    # Applying our expert rules on top of the AI
    sarcasm = detect_sarcasm(text)
    final_sentiment = "negative" if sarcasm else SENTIMENT_LABELS[np.argmax(s_probs)]

    ml_urgency = URGENCY_LABELS[np.argmax(u_probs)]
    final_urgency = apply_safety_override(text, ml_urgency)

    return {
        "original_text": text,
        "sentiment": final_sentiment.upper(),
        "urgency": final_urgency.upper(),
        "categories": [ALL_CATEGORIES[i].upper() for i, p in enumerate(c_probs) if p >= 0.5],
        "overrides": {"sarcasm": sarcasm, "safety": final_urgency == "high" and ml_urgency != "high"}
    }

## 3. The Hybrid Inference Pipeline
This is the core 'intelligent' component. It combines the ML model outputs with:
- **Safety Overrides**: Forcing high urgency on danger keywords.
- **Sarcasm Detection**: Reversing sentiment based on contextual contradictions.

In [ ]:
import shutil

shutil.make_archive('category_model', 'zip', 'saved_model_category')
shutil.make_archive('sentiment_model', 'zip', 'saved_model_sentiment')
shutil.make_archive('urgency_model', 'zip', 'saved_model_urgency')

print("Zipping complete!")

## 4. Model Packaging
We archive the trained model weights and configuration files into ZIP formats for easier deployment.

In [ ]:
import shutil
import os
from google.colab import files

print("Zipping models...")

shutil.make_archive("complaint_models", "zip", root_dir=".",
                    base_dir=None,
                    verbose=True)

folders = ["saved_model_sentiment", "saved_model_urgency", "saved_model_category"]

for folder in folders:
    if os.path.exists(folder):
        shutil.make_archive(folder, "zip", root_dir=".", base_dir=folder)
        print(f"✓ Zipped: {folder}.zip")
    else:
        print(f"✗ Not found: {folder} — make sure training completed")
print("\nStarting downloads...")
for folder in folders:
    zip_name = f"{folder}.zip"
    if os.path.exists(zip_name):
        print(f"Downloading {zip_name}...")
        files.download(zip_name)

print("\nDone! Check your browser's download folder.")


## 5. Download Artifacts
Finally, we trigger the browser download for the zipped model files.